# RG3 gas: randomization stability as ensemble confirmation
Five fixed seeds, each60 trees per view, existing jitter4/residual4 inputs. No new physical feature and no seed selected using test labels.
Mean, median, lower quartile, and mean minus uncertainty penalties compared. This is empirical model randomization spread, NOT a statistical confidence interval or calibrated probability.
Original baseline200 preserved; this is a 5x60-tree ensemble. Inner group-held-out normal99% thresholds and train-only selection.
Repeated-data exploratory experiment; low spread is not proof of a correct gas diagnosis.

In [ ]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix,average_precision_score
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
seeds=[42,7,101,2024,91]
memo={}
def predict(ref,q):
    key=(tuple(ref),tuple(q))
    if key in memo:return memo[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    if len(np.unique(y[ref]))<2:return np.zeros((len(q),len(seeds)))
    rr=np.full((len(ref),4),np.nan)
    for b in groups.iloc[ref].unique():
        mask=groups.iloc[ref].eq(b).to_numpy()
        if (~mask).any() and (y[ref[~mask]]==0).any():rr[mask]=res_features(ref[~mask],ref[mask])
    views=[(d.iloc[ref][stdcols].to_numpy(float),d.iloc[q][stdcols].to_numpy(float)),(rr,res_features(ref,q))]
    result=[]
    for seed in seeds:
        view_scores=[]
        for tr,te in views:
            m=make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),RandomForestClassifier(n_estimators=60,max_depth=8,min_samples_leaf=3,class_weight='balanced_subsample',random_state=seed,n_jobs=2))
            view_scores.append(m.fit(tr,y[ref]).predict_proba(te)[:,1])
        result.append(np.max(view_scores,axis=0))
    memo[key]=np.column_stack(result)
    return memo[key]
def combine(x):
    return {'seed_mean':x.mean(axis=1),'seed_median':np.median(x,axis=1),'seed_lower_quartile':np.quantile(x,.25,axis=1),'mean_minus_half_std':x.mean(axis=1)-.5*x.std(axis=1),'mean_minus_std':x.mean(axis=1)-x.std(axis=1)}
records=[];choices=[];uncertainty=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,b in enumerate(blocks):
        q=np.flatnonzero(groups.eq(b));ref=np.flatnonzero(~groups.eq(b)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        if len(np.unique(y[ref]))<2:continue
        outer=predict(ref,q);inner=np.full((len(ref),len(seeds)),np.nan)
        for ib in groups.iloc[ref].unique():
            mask=groups.iloc[ref].eq(ib).to_numpy()
            if (~mask).any():inner[mask]=predict(ref[~mask],ref[mask])
        cal=combine(inner);out=combine(outer);cs=[]
        for name,v in cal.items():
            norm=v[(y[ref]==0)&np.isfinite(v)]
            cut=float(np.quantile(norm,.99,method='higher')) if len(norm) else 1.
            valid=np.isfinite(v);pr=v[valid]>cut
            tn,fp,fn,tp=confusion_matrix(y[ref][valid],pr,labels=[0,1]).ravel()
            f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.
            cs.append(dict(model=name,threshold=cut,inner_F1=f1,inner_FP=int(fp)))
            for k,i in enumerate(q):records.append(dict(scheme=scheme,model=name,TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(out[name][k]),threshold=cut,prediction=int(out[name][k]>cut)))
        selected=max(cs,key=lambda c:(c['inner_F1'],-c['inner_FP'],c['model']=='seed_mean'))
        for k,i in enumerate(q):
            records.append(dict(scheme=scheme,model='inner_selected',TimeStamp=d.TimeStamp.iloc[i],block=b,actual=int(y[i]),score=float(out[selected['model']][k]),threshold=selected['threshold'],prediction=int(out[selected['model']][k]>selected['threshold'])))
            uncertainty.append(dict(scheme=scheme,TimeStamp=d.TimeStamp.iloc[i],actual=int(y[i]),seed_mean=float(outer[k].mean()),seed_std=float(outer[k].std()),**{f'seed_{s}':float(outer[k,l]) for l,s in enumerate(seeds)}))
        choices.append(dict(scheme=scheme,block=b,**selected))
        print('Finished uncertainty ensemble',scheme,b,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,name),g in pp.groupby(['scheme','model'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,AP=average_precision_score(g.actual,g.score)))
metrics=pd.DataFrame(rows)
pp.to_csv(tables/'rg3_uncertainty_ensemble_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(tables/'rg3_uncertainty_ensemble_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(uncertainty).to_csv(tables/'rg3_uncertainty_ensemble_seed_scores.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(tables/'rg3_uncertainty_ensemble_inner_choices.csv',index=False,encoding='utf-8-sig')
print(metrics.to_string(index=False))


## Executed results
Internal selected whole TP4 FP9 FN13 TN557 F1=0.267; forward TP2 FP4 FN11 TN423 F1=0.211. No improvement. Random-seed spread is not a statistical confidence interval.
See reports/rg3_iterative_improvement_log.md for protocol, limitations and source tables.